# SRC-002 — CoreOps Data Audit

## 1. Dataset Inventory

### Objective

Identify the available files, formats, sizes, and directory structure
of the selected CoreOps-22NFs dataset.

### Scope

- File discovery and classification.
- File counts by format.
- Storage distribution.
- Identification of missing or unexpected files.

This stage does not inspect the internal contents of the dataset.

In [1]:
from daicp.offline.data import inventory as di
from daicp.offline.data.visualization import DataVisualization
from daicp.offline.data.profiling import Schema, Quality
from daicp.offline.data.dataset_tools import CoreOps

import pandas as pd
import json

In [2]:
inventory = di.DataInventory("SRC-002")

print(inventory.raw_data_path)
print(inventory.discover_file_extensions())

/Users/emiliogarcialopez/projectGitHub/distributed-agentic-ai-platform/data/raw/SRC-002_coreops
{'.csv', '.tag', '.lock', '.json', '.yaml', '.log', '.metadata'}


In [3]:
files = inventory.file_inventory()
fig = DataVisualization.plot_file_distribution(files)
fig.show()

## 2. Schema Inspection

### Objective

Identify the data structures, fields, data types, and formats
available in SRC-002.

### Scope

- JSON structure and nested fields.
- CSV columns and inferred data types.
- Operational log format.
- Timestamp representations.

Only representative records will be inspected to avoid unnecessary
processing of large files.

In [4]:
schema = Schema(nrows=50)
root = inventory.raw_data_path / "CoreOps-22NFs"

# 2.1 Ground truth
display(schema.inspect_json_schema(root / "labels/fault_labels.json", "run_history"))

# 2.2 Per-NF metrics
for nf in ["pcf", "amf-a1", "smf-e1"]:
    path = root / f"metrics/per_nf0525/{nf}_20260518_142750.csv"

    print(f"\n{nf.upper()} Metrics")
    display(schema.inspect_csv_schema(path, sep=";"))

# 2.3 Fault windows
windows = pd.read_json(root / "metrics/aligned_fault_window.json")
display(windows.dtypes.to_frame(name="dtype"))

# 2.4 Special formats
for file in ["metrics/global_matrix.csv", "logs/pcf.log"]:
    print(f"\n{file}")
    print("\n".join(schema.preview_text(root / file)))

{'root_fields': ['schema_version', 'description', 'run_history'],
 'record_fields': ['fault_id',
  'run_id',
  'targets',
  'fault_type',
  'fault_category',
  'start_time',
  'end_time',
  'duration_sec',
  'params',
  'status',
  'mitigation_hint',
  'overlap_possible']}


PCF Metrics


,column,dtype
0,timestamp,str
1,scrape_ok,int64
2,restart_detected,int64
3,kernel_tcp_ArpFilter,int64
4,kernel_tcp_ArpFilter_rate,float64
...,...,...
273,socket_CLOSE_WAIT,int64
274,socket_ESTAB,int64
275,socket_SYN_SENT,int64
276,socket_TIME_WAIT,int64



AMF-A1 Metrics


,column,dtype
0,timestamp,str
1,scrape_ok,int64
2,restart_detected,int64
3,kernel_tcp_ArpFilter,int64
4,kernel_tcp_ArpFilter_rate,float64
...,...,...
273,socket_CLOSE_WAIT,int64
274,socket_ESTAB,int64
275,socket_SYN_SENT,int64
276,socket_TIME_WAIT,int64



SMF-E1 Metrics


,column,dtype
0,timestamp,str
1,scrape_ok,int64
2,restart_detected,int64
3,kernel_tcp_ArpFilter,int64
4,kernel_tcp_ArpFilter_rate,float64
...,...,...
273,socket_CLOSE_WAIT,int64
274,socket_ESTAB,int64
275,socket_SYN_SENT,int64
276,socket_TIME_WAIT,int64


,dtype
fault_id,str
fault_type,str
targets,object
start_time,"datetime64[us, UTC+08:00]"
end_time,"datetime64[us, UTC+08:00]"



metrics/global_matrix.csv
timestamp;data
2026-05-18 14:27:52;amf-a2=UP;amf-a1=UP;smf-u1=UP;smf-u2=UP;pcf=UP;upf-m1=UP;upf-e2=UP;upf-e1=UP;amf-b2=UP;amf-b1=UP;smf-m1=UP;upf-u2=UP;upf-u1=UP;smf-e1=UP;smf-e2=UP;amf-c1=UP
2026-05-18 14:28:16;amf-a2=UP;amf-a1=UP;smf-u1=UP;smf-u2=UP;pcf=UP;upf-m1=UP;upf-e2=UP;upf-e1=UP;amf-b2=UP;amf-b1=UP;smf-m1=UP;upf-u2=UP;upf-u1=UP;smf-e1=UP;smf-e2=UP;amf-c1=UP

logs/pcf.log
Open5GS daemon v2.7.7+

05/14 16:10:59.329: [app] INFO: Configuration: '/etc/open5gs/pcf.yaml' (../lib/app/ogs-init.c:144)


### Schema Inspection Findings

- Ground-truth labels contain structured fault metadata.
- Fault windows provide timezone-aware timestamps (UTC+08:00).
- Per-NF metrics use semicolon-separated CSV files.
- Global availability records contain timestamped NF status values.
- Open5GS logs contain operational events with timestamps, modules,
  severity levels, and messages.
- Log timestamps do not explicitly include the year or timezone.

**Decision:** Data formats are suitable for further analysis.
Timestamp normalization and specialized parsing of the global
availability matrix are required.

## 3. Data Quality Profiling

### Objective

Assess the completeness, uniqueness, and validity of the selected
CoreOps datasets.

### Scope

- Missing values.
- Duplicate records and identifiers.
- Timestamp validity.
- Fault duration consistency.
- Per-NF metrics quality.

Operational log analysis and cross-source temporal consistency
are addressed in subsequent sections.

In [5]:
report = CoreOps.audit_coreops_quality(root)

for name, result in report.items():
    print(f"\n{name.upper()}")
    display(result)


SUMMARY


,dataset,rows,columns,missing_values,missing_pct,duplicates
0,Fault Labels,160,29,2260,48.71,0
1,Fault Windows,160,5,0,0.00,0
2,amf-a1_20260518_142750,18911,278,45650,0.87,0
3,pcf_20260518_142750,18911,278,25300,0.48,0
4,smf-e1_20260518_142750,18910,278,0,0.00,0



VALIDATIONS


,check,invalid_records
0,Missing fault IDs,0
1,Missing targets,0
2,Invalid timestamps,0
3,Invalid intervals,0
4,Invalid durations,0
5,Duration mismatches,16



MISSING_FIELDS


,column,missing_pct
0,params.bandwidth_kbps,97.500
1,params.delay_ms,97.500
2,params.mem_mb,96.875
3,params.cpu_workers,96.250
4,params.base_delay_ms,95.000
5,params.jitter_ms,95.000
6,params.safe_limit,93.125
7,params.tool,93.125
8,params.method,90.000
9,params.direction,90.000


In [6]:
faults = pd.json_normalize(
    json.loads((root / "labels/fault_labels.json").read_text(encoding="utf-8"))["run_history"]
)

start = pd.to_datetime(faults["start_time"], utc=True)
end = pd.to_datetime(faults["end_time"], utc=True)

faults["calculated_duration"] = (end - start).dt.total_seconds()
faults["duration_diff"] = faults["calculated_duration"] - faults["duration_sec"]

display(
    faults.loc[
        faults["duration_diff"].abs() > 1,
        ["fault_id", "fault_type", "duration_sec", "calculated_duration", "duration_diff"]
    ]
)

,fault_id,fault_type,duration_sec,calculated_duration,duration_diff
3,fault_20260518172632_9160,nf_blackhole,321,325.0,4.0
19,fault_20260519035116_689,nf_blackhole,437,440.0,3.0
28,fault_20260519091509_7510,nf_blackhole,749,753.0,4.0
30,fault_20260519103152_9106,nf_blackhole,603,606.0,3.0
34,fault_20260519131419_20892,nf_blackhole,338,341.0,3.0
52,fault_20260520003347_15094,nf_blackhole,947,950.0,3.0
57,fault_20260520035528_16616,nf_blackhole,425,428.0,3.0
73,fault_20260520141645_5662,nf_blackhole,453,456.0,3.0
74,fault_20260520145313_10980,nf_blackhole,734,737.0,3.0
82,fault_20260520204303_5360,nf_blackhole,435,439.0,4.0


### Data Quality Findings

- The dataset contains 160 fault records with unique identifiers.
- No invalid timestamps or chronological intervals were detected.
- Missing ground-truth values primarily correspond to optional,
  fault-specific parameters.
- Selected NF metrics exhibit low overall missing-value rates.
- Sixteen fault records exhibit duration discrepancies of 3–4 seconds.
- All sixteen discrepancies correspond to `nf_blackhole`.
- Source-code inspection identified an additional three-second
  recovery delay after removing the traffic-blocking rule.
- The discrepancies are consistent with the injection implementation
  and do not require modification of the original labels.

**Decision: WARN — Explained temporal discrepancies.**

The dataset remains suitable for subsequent analysis.
Fault timestamps should not be interpreted as exact boundaries
of observable network disruption.

## 4. Ground Truth Analysis

### Objective

Characterize the fault-injection ground truth through descriptive
statistics and interactive visualizations.

### Scope

- Fault type distribution.
- Affected network function frequency.
- Fault duration distribution.
- General incident statistics.

Ground-truth labels are used exclusively for dataset analysis
and evaluation, not as agent-visible diagnostic evidence.

In [7]:
faults = CoreOps.load_faults(root)

display(faults[["fault_type", "duration_sec"]].describe(include="all"))

DataVisualization.plot_category_frequency(
    faults, "fault_type", "Fault Type Distribution"
).show()

DataVisualization.plot_category_frequency(
    faults, "targets", "Affected Network Functions"
).show()

DataVisualization.plot_numeric_by_category(
    faults, "fault_type", "duration_sec", "Fault Duration by Type"
).show()

,fault_type,duration_sec
count,160,160.000000
unique,10,NaN
top,link_disconnect,NaN
freq,39,NaN
mean,NaN,710.218750
std,NaN,282.751945
min,NaN,252.000000
25%,NaN,433.250000
50%,NaN,727.500000
75%,NaN,953.750000


### Ground Truth Analysis Findings

- The dataset contains 160 fault-injection records across 10 fault types.
- Network faults represent 52.50% of incidents, followed by NF anomalies
  (40.63%) and resource contention (6.88%).
- The distribution is imbalanced, with `link_disconnect` representing
  the most frequent fault type (24.38%).
- Fault injections target 19 distinct network functions.
- PCF (15), AMF-B1 (14), and AMF-A1 (13) are the most frequently targeted NFs.
- The target distribution is heterogeneous, with some NFs represented
  by fewer than five incidents.
- Recorded fault durations range from 252 to 1,196 seconds,
  with a mean duration of 710.22 seconds.
- Duration distributions vary across fault types.

**Decision: PASS with limitations.**

SRC-002 provides sufficient ground-truth diversity for initial RCA
experiments. However, class imbalance and heterogeneous NF coverage
must be considered during scenario selection and evaluation.

Operational evidence availability remains to be verified.

## 5. Operational Evidence Analysis

### Objective

Characterize the operational evidence available in CoreOps-22NFs.

### Scope

- Log severity distribution across network functions.
- Network function availability observations.
- Per-NF metric coverage and variability.

This section characterizes the available operational evidence.
Causal relationships with injected faults are evaluated separately.

In [8]:
logs = CoreOps.summarize_logs(root)
availability = CoreOps.summarize_availability(root)
metrics = CoreOps.summarize_metrics(root)

display(logs.sort_values("warning_error_pct", ascending=False))
display(availability.sort_values("down_pct", ascending=False))
display(metrics)

DataVisualization.plot_percentage_by_category(
    logs, "nf", "warning_error_pct", "Log Warning/Error Distribution"
).show()

DataVisualization.plot_percentage_by_category(
    availability, "nf", "down_pct", "Network Function DOWN Observations"
).show()

,nf,events,info,warnings,errors,warning_error_pct
18,upf-u1,3343,267,1707,1369,92.01
19,upf-u2,1443,138,732,573,90.44
14,udr,8834,1077,3747,4010,87.81
16,upf-e2,11083,1716,6624,2743,84.52
13,udm,9121,2185,3334,3602,76.04
5,ausf,12228,3329,4410,4489,72.78
15,upf-e1,5245,1752,1985,1508,66.60
9,smf-e2,33713,11965,12929,8819,64.51
7,pcf,16787,6935,4582,5270,58.69
11,smf-u1,22862,10009,6700,6153,56.22


,nf,observations,up,down,other_states,down_pct
0,amf-a1,18910,18744,166,0,0.88
5,pcf,18910,18818,92,0,0.49
7,smf-e2,18910,18825,85,0,0.45
3,amf-b2,18910,18830,80,0,0.42
2,amf-b1,18910,18855,55,0,0.29
12,upf-e2,18910,18862,48,0,0.25
4,amf-c1,18910,18882,28,0,0.15
9,smf-u1,18910,18882,28,0,0.15
15,upf-u2,18910,18883,27,0,0.14
11,upf-e1,18910,18887,23,0,0.12


,nf,rows,columns,numeric_columns,variable_numeric_columns,scrape_failures,restart_flags
0,amf-a1,18911,278,277,35,166,10
1,pcf,18911,278,277,31,92,10
2,smf-e1,18910,278,277,29,0,1


### Operational Evidence Analysis Findings

- Operational logs are available for multiple 5G Core network functions.
- High proportions of WARNING and ERROR events were observed in several NFs.
- Log severity alone does not establish fault causality.
- The global availability matrix contains 18,910 observations per monitored NF.
- Several network functions exhibit DOWN observations, while others remain UP.
- Selected per-NF metric files contain 278 columns each.
- Only 29–35 numeric columns exhibit variability in the inspected metric files.
- Scrape failures were detected in AMF-A1 (166) and PCF (92).
- The number of scrape failures exactly matches the number of DOWN
  observations for AMF-A1, PCF, and SMF-E1.
- This suggests a possible relationship between metric collection
  failures and the reported availability states.
- Timestamp-level verification is required before confirming this relationship.

**Decision: PASS with limitations.**

Operational evidence is available for further RCA investigation.
However, metric collection failures, availability states, and actual
network function failures must be distinguished.

## 6. Cross-Source Consistency

### Objective

Verify temporal consistency and evidence coverage between
fault labels, fault windows, operational logs, and NF metrics.

### Scope

- Ground-truth and fault-window correspondence.
- Operational log coverage.
- Global NF availability coverage.
- Per-NF metric availability.
- Successful metric collection during fault windows.

Temporal correspondence does not establish fault causality.

In [9]:
consistency = CoreOps.check_temporal_consistency(root)

display(consistency.head(15))

,fault_id,fault_type,nf,label_window_match,log_events,global_observations,global_down,metric_samples,successful_scrapes
0,fault_20260518152205_22929,link_disconnect,pcf,True,18,23,0,23,23
1,fault_20260518160615_3460,link_disconnect,nrf,True,3,<NA>,<NA>,<NA>,<NA>
2,fault_20260518164855_14817,link_disconnect,upf-e1,True,0,21,0,<NA>,<NA>
3,fault_20260518172632_9160,nf_blackhole,ausf,True,411,<NA>,<NA>,<NA>,<NA>
4,fault_20260518175704_16126,nf_freeze,pcf,True,0,23,23,23,0
5,fault_20260518183831_4478,nf_freeze,amf-a1,True,54,28,28,28,0
6,fault_20260518192245_32080,link_disconnect,amf-a2,True,0,33,0,<NA>,<NA>
7,fault_20260518200400_8911,nf_freeze,nrf,True,0,<NA>,<NA>,<NA>,<NA>
8,fault_20260518204102_25415,link_loss,upf-e1,True,0,25,0,<NA>,<NA>
9,fault_20260518212059_10788,link_loss,pcf,True,0,16,0,16,16


In [10]:
summary = pd.DataFrame({
    "Check": [
        "Total incidents",
        "Matching fault windows",
        "Incidents with target logs",
        "Incidents with target log events",
        "Incidents covered by global matrix",
        "Incidents with target NF metrics",
        "Incidents with successful metric scrapes"
    ],
    "Count": [
        len(consistency),
        consistency["label_window_match"].sum(),
        consistency["log_events"].notna().sum(),
        consistency["log_events"].gt(0).sum(),
        consistency["global_observations"].gt(0).sum(),
        consistency["metric_samples"].notna().sum(),
        consistency["successful_scrapes"].gt(0).sum()
    ]
})

display(summary)

,Check,Count
0,Total incidents,160
1,Matching fault windows,160
2,Incidents with target logs,160
3,Incidents with target log events,56
4,Incidents covered by global matrix,122
5,Incidents with target NF metrics,34
6,Incidents with successful metric scrapes,26


### Cross-Source Consistency Findings

- All 160 fault records match their corresponding fault windows.
- Operational log files are available for all target network functions.
- Only 56 incidents (35.00%) contain target-NF log events within
  their recorded fault windows.
- Global availability observations are available for 122 incidents
  (76.25%).
- Selected per-NF metric files are available for the target functions
  of 34 incidents (21.25%).
- Successful metric scrapes were observed for 26 incidents (16.25%).
- Some incidents exhibit operational effects consistent with
  their injected fault types.
- Temporal coverage does not establish causal relationships.
- Availability states and scrape failures may represent related
  observations rather than independent evidence.

**Decision: WARN — Partial operational evidence coverage.**

The dataset is suitable for controlled RCA experiments using
selected incidents with sufficient supporting evidence.

Additional per-NF metrics may be required to expand evaluation
coverage.

## 7. Audit Dashboard

### Objective

Consolidate the principal data quality, ground-truth, and
operational evidence indicators into an interactive dashboard.

The dashboard summarizes the available data and its limitations.
It does not establish causal relationships between injected
faults and operational observations.

In [11]:
dashboard = DataVisualization.plot_audit_dashboard(
    faults, logs, availability, consistency
)

dashboard.show()

In [12]:
output = inventory.project_root / "data/interim/SRC-002_coreops"
output.mkdir(parents=True, exist_ok=True)

dashboard.write_html(
    output / "audit_dashboard.html",
    include_plotlyjs=True
)

## 8. Findings & Suitability Decision

### 8.1 Dataset Summary

| Indicator | Result |
|---|---|
| Total Incidents | 160 |
| Fault Types | 10 |
| Target Network Functions | 19 |
| Fault Duration Range | 252–1,196 seconds |
| Mean Fault Duration | 710.22 seconds |
| Duplicate Fault IDs | 0 |
| Matching Fault Windows | 160/160 (100%) |

### 8.2 Data Quality

- No duplicate fault identifiers or invalid chronological intervals
  were detected.
- Missing values in fault labels primarily correspond to optional,
  fault-specific parameters.
- Selected NF metrics exhibit low overall missing-value rates.
- Sixteen `nf_blackhole` incidents contain duration differences of
  3–4 seconds, consistent with additional recovery operations in
  the published fault-injection implementation.
- Original records remain unchanged.

**Status: PASS with documented limitations.**

### 8.3 Ground Truth

- Ten fault types are represented across three operational fault families.
- Network faults account for 52.50% of incidents.
- NF anomalies account for 40.63%.
- Resource contention accounts for 6.88%.
- The distribution is imbalanced, particularly for resource-related faults.
- Fault injections target 19 distinct network functions.

**Status: PASS with class imbalance.**

### 8.4 Operational Evidence

| Evidence | Incidents | Coverage |
|---|---:|---:|
| Matching Fault Windows | 160 | 100.00% |
| Available Target Log Files | 160 | 100.00% |
| Target Log Events During Faults | 56 | 35.00% |
| Global Availability Observations | 122 | 76.25% |
| Available Target NF Metric Files | 34 | 21.25% |
| Successful Metric Scrapes | 26 | 16.25% |

- Operational logs and availability observations provide evidence
  suitable for investigating selected incidents.
- Several incidents contain no target-NF log events during their
  recorded fault windows.
- Individual NF metrics are currently available for only three
  selected network functions.
- DOWN observations and scrape failures may represent related
  measurements rather than independent evidence.
- Temporal coverage does not establish fault causality.

**Status: WARN — Partial operational evidence coverage.**

### 8.5 Limitations and Risks

| Risk | Implication | Mitigation |
|---|---|---|
| Class imbalance | Biased aggregate RCA evaluation | Report metrics by fault family |
| Partial NF metrics | Limited incident coverage | Select supported scenarios |
| Sparse operational logs | Insufficient evidence for some incidents | Consider related NF logs |
| Timestamp assumptions | Potential temporal misalignment | Verify timezone interpretation |
| Injection overhead | Approximate fault boundaries | Preserve timing uncertainty |
| Ground-truth leakage | Inflated RCA performance | Keep labels separate from agent inputs |
| Synthetic environment | Limited production generalization | Restrict experimental conclusions |

### 8.6 Suitability Decision

**Decision: CONDITIONALLY ACCEPTED**

SRC-002 CoreOps is suitable for controlled development and experimental
evaluation of evidence-grounded Root Cause Analysis in 5G Core networks.

The dataset provides:

- Structured fault-injection ground truth.
- Operational logs from multiple network functions.
- Network function availability observations.
- Selected per-NF telemetry.
- Experimental infrastructure context.

However, the complete set of 160 incidents should not be treated
as fully validated RCA evaluation cases.

Incident selection must consider operational evidence availability,
temporal coverage, fault diversity, and potential data leakage.

### 8.7 Next Steps

The initial RCA evaluation should prioritize incidents with
sufficient operational evidence, particularly:

- PCF network connectivity failures.
- PCF or AMF availability failures.
- SMF resource contention scenarios.

Further investigation will determine whether additional operational
metrics or related NF logs are required.

No additional general-purpose data auditing is required at this stage.